In [ ]:
!pip install groq

In [2]:
import os
import json
from groq import Groq

In [3]:
from google.colab import userdata

In [4]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
key = os.environ["GROQ_API_KEY"]
print(f"API key loaded successfully — length: {len(key)} characters")

API key loaded successfully — length: 56 characters


In [5]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [16]:
SYSTEM_PROMPT = """
You are an expert fraud analyst with 15 years of experience detecting
scams across SMS, email, WhatsApp, and social media.

## CRITICAL RULE — EMAIL DOMAIN ANALYSIS:
When analyzing any email, you MUST examine the sender's email domain
with extreme suspicion. Follow this logic:

1. Extract the domain from the email address (everything after @)
2. Ask: does this domain match a well-known, established company?
3. Ask: is this a free email service? (gmail, yahoo, hotmail = red flag)
4. Ask: does the subdomain prefix look auto-generated or fake?
   Examples of suspicious patterns:
   - hralert@, hr-team@, noreply-hr@, jobs-alert@
   - These are created by scammers to sound official
5. Ask: for a UAE company of any size, would they use this exact domain?
   A real Wadi Al Salam Group would use @wadialsalamgroup.com or
   a well-established .ae domain — NOT a generic alert subdomain

## RED FLAGS SPECIFIC TO JOB SCAMS:
- Email domains registered recently or with no web presence
- "HR alert" style prefixes (hralert@, hr.alert@, hrteam@)
- Asking for documents, fees, or bank details before an in-person interview
- Generic congratulatory language ("We are pleased to inform you...")
- No direct phone number for a named HR manager
- Interview scheduled suspiciously fast after no prior application
- Salary/benefits mentioned upfront without discussing role requirements

## THINK STEP BY STEP:
1. Read the full message
2. If an email address is present, analyze the domain critically
3. Identify red flags AND safe indicators
4. Assign confidence score
5. Return structured JSON

YOU MUST RESPOND WITH ONLY VALID JSON using exactly this structure:

{
  "is_scam": true or false,
  "confidence": 0 to 100,
  "scam_type": one of ["phishing", "lottery_fraud", "advance_fee",
               "romance_scam", "investment_fraud", "impersonation",
               "job_scam", "tech_support_scam", "not_a_scam", "unknown"],
  "domain_analysis": {
    "domain_found": "the domain extracted from email/url or null",
    "domain_suspicious": true or false,
    "reason": "why the domain is or is not suspicious"
  },
  "red_flags": ["list", "of", "red", "flags"],
  "safe_flags": ["list", "of", "safe", "indicators"],
  "explanation": "Clear 2-3 sentence explanation for a non-technical person",
  "recommended_action": "What the recipient should do"
}
"""

In [11]:
def analyze_message(message: str) -> dict:
    """
    Sends a message to the scam classifier agent.
    Returns a structured verdict as a Python dictionary.
    """
    print(f"\n{'='*55}")
    print(f"ANALYZING MESSAGE...")
    print(f"{'='*55}")
    print(f"Message: {message[:100]}{'...' if len(message) > 100 else ''}")
    print(f"{'='*55}\n")

    # CONCEPT: The messages list is the conversation history.
    # 'system' = the agent's role and instructions
    # 'user'   = the actual input we want analyzed
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",   # Free, powerful model on Groq
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"Analyze this message:\n\n{message}"
            }
        ],
        temperature=0.1,    # Low temperature = more consistent, less creative
                            # For classifiers, we want predictable output
        max_tokens=800,     # Enough for our JSON + reasoning
    )

    # Extract the raw text response from the LLM
    raw_response = response.choices[0].message.content

    # CONCEPT: LLMs sometimes add markdown fences like ```json ... ```
    # We strip those so we can parse clean JSON
    raw_response = raw_response.strip()
    if raw_response.startswith("```"):
        raw_response = raw_response.split("```")[1]
        if raw_response.startswith("json"):
            raw_response = raw_response[4:]
    raw_response = raw_response.strip()

    # Parse JSON string → Python dictionary
    # If parsing fails, we return a safe error dict instead of crashing
    try:
        result = json.loads(raw_response)
    except json.JSONDecodeError:
        print("Warning: LLM returned non-JSON. Raw response:")
        print(raw_response)
        result = {
            "is_scam": None,
            "confidence": 0,
            "scam_type": "unknown",
            "red_flags": [],
            "safe_flags": [],
            "explanation": "Could not parse response",
            "recommended_action": "Review manually",
            "raw_response": raw_response
        }

    return result

In [8]:
def display_verdict(result: dict):
    """Prints the agent's verdict in a readable format."""

    verdict = "🚨 SCAM DETECTED" if result.get("is_scam") else "✅ LIKELY SAFE"
    confidence = result.get("confidence", 0)

    # Confidence bar (visual representation)
    filled = int(confidence / 10)
    bar = "█" * filled + "░" * (10 - filled)

    print(f"\n{'─'*55}")
    print(f"  VERDICT: {verdict}")
    print(f"  Confidence: [{bar}] {confidence}%")
    print(f"  Scam Type:  {result.get('scam_type', 'N/A').replace('_', ' ').title()}")
    print(f"{'─'*55}")

    if result.get("red_flags"):
        print(f"\n  🚩 Red Flags Found:")
        for flag in result["red_flags"]:
            print(f"     • {flag}")

    if result.get("safe_flags"):
        print(f"\n  ✔  Safe Indicators:")
        for flag in result["safe_flags"]:
            print(f"     • {flag}")

    print(f"\n  📋 Explanation:")
    print(f"     {result.get('explanation', '')}")

    print(f"\n  💡 Recommended Action:")
    print(f"     {result.get('recommended_action', '')}")
    print(f"{'─'*55}\n")


In [9]:
test_messages = [

    # Example 1: Classic UAE lottery scam
    """Congratulations! You have won AED 500,000 in the Dubai
    Government Lucky Draw. To claim your prize, send your
    Emirates ID and pay AED 250 processing fee to account
    AE12 0030 0000 0000 0000. Offer expires in 24 hours!""",

    # Example 2: Phishing impersonating a bank
    """Dear Customer, your Emirates NBD account has been
    temporarily suspended due to suspicious activity.
    Click here immediately to verify: http://emiratesnbd-secure.tk/login
    Failure to verify within 2 hours will result in permanent closure.""",

    # Example 3: Legitimate message (should NOT be flagged)
    """Hi Sara, this is Ahmed from HR. Your interview is confirmed
    for Monday 10am at our Dubai Internet City office.
    Please bring your original documents. See you then!""",

    # Example 4: Job scam (common in UAE)
    """URGENT HIRING! Work from home, earn AED 5000/week.
    No experience needed. WhatsApp us now: +971XXXXXXXXX.
    Limited slots available. Send your name and bank details to apply.""",

]

In [12]:
print("\n" + "="*55)
print("  ANTI-SCAM AGENT — PHASE 1")
print("  Powered by Llama 3 (Groq Free Tier)")
print("="*55)

results = []   # Store all results for later analysis

for i, message in enumerate(test_messages, 1):
    print(f"\n[TEST {i} of {len(test_messages)}]")
    result = analyze_message(message)
    display_verdict(result)
    results.append(result)

    # CONCEPT: Rate limiting — Groq free tier allows ~30 requests/min
    # Adding a small pause between calls avoids hitting limits
    import time
    time.sleep(1)


  ANTI-SCAM AGENT — PHASE 1
  Powered by Llama 3 (Groq Free Tier)

[TEST 1 of 4]

ANALYZING MESSAGE...
Message: Congratulations! You have won AED 500,000 in the Dubai 
    Government Lucky Draw. To claim your pri...


───────────────────────────────────────────────────────
  VERDICT: 🚨 SCAM DETECTED
  Confidence: [█████████░] 95%
  Scam Type:  Lottery Fraud
───────────────────────────────────────────────────────

  🚩 Red Flags Found:
     • unsolicited prize
     • request for personal data
     • request for payment
     • urgency

  📋 Explanation:
     This message is likely a scam because it asks for personal and financial information in exchange for a prize, and creates a sense of urgency to prompt a quick response. Legitimate lotteries do not ask for payment to claim a prize. The request for Emirates ID and payment is a clear attempt to steal your identity and money.

  💡 Recommended Action:
     Do not respond to the message, do not send any money or personal data, and report th

In [13]:
print("\n" + "="*55)
print("  BATCH ANALYSIS SUMMARY")
print("="*55)

scams_found = sum(1 for r in results if r.get("is_scam"))
safe_found = sum(1 for r in results if r.get("is_scam") == False)
avg_confidence = sum(r.get("confidence", 0) for r in results) / len(results)

print(f"  Messages analyzed : {len(results)}")
print(f"  Scams detected    : {scams_found}")
print(f"  Safe messages     : {safe_found}")
print(f"  Avg confidence    : {avg_confidence:.1f}%")
print("="*55)


  BATCH ANALYSIS SUMMARY
  Messages analyzed : 4
  Scams detected    : 3
  Safe messages     : 1
  Avg confidence    : 93.8%


In [14]:
your_message = '''
WAS Group <hralert@wadialsagroup.com>
Apr 21, 2026, 6:51 PM (9 days ago)
to me

Dear Khan

Thank you for your interest in joining the WAS Group of Companies. We have reviewed your Teacher Coordinator application and are pleased to invite you for an in-person interview to discuss your background and how you might fit within our team.

Interview Schedule

Date: Wednesday, 22nd April 2026

Time: Between 09:00 AM and 12:00 PM

Location: Office M-06, M Floor, Silver Building (Building No. 10), Abu Hail, Dubai

Landmark: Next to Shaklan Market

By Metro: Abu Hail Metro Station (Exit 2).

Entrance: Please use the office entrance of the building housing "For You Cafe."

Maps: https://maps.app.goo.gl/uY92kPrunNDLNzWN8

What to Bring Please bring a printed copy of your updated CV and any relevant professional certifications or documents for our review.

Company Benefits As part of our commitment to our employees, we provide a residency visa, accommodation, and transportation in full compliance with UAE labor regulations.

We look forward to meeting with you.

Best regards,

Hessa Al-Falasi Human Resources Manager

WAS Group of Companies


'''

In [17]:
your_result = analyze_message(your_message)
display_verdict(your_result)


ANALYZING MESSAGE...
Message:  
WAS Group <hralert@wadialsagroup.com>
Apr 21, 2026, 6:51 PM (9 days ago)
to me

Dear Khan

Thank y...


───────────────────────────────────────────────────────
  VERDICT: 🚨 SCAM DETECTED
  Confidence: [████████░░] 80%
  Scam Type:  Job Scam
───────────────────────────────────────────────────────

  🚩 Red Flags Found:
     • Similar but not exact match to a well-known company domain
     • Lack of direct phone number for the HR manager
     • Interview scheduled without prior application or discussion
     • Generic congratulatory language
     • Request for documents and certifications without a secure channel

  ✔  Safe Indicators:
     • Specific interview location and time provided
     • Maps link provided for location
     • Company benefits mentioned

  📋 Explanation:
     This email has several red flags that suggest it might be a job scam, including a suspicious domain name and a lack of direct contact information for the HR manager. While the e